## Data Cleaning & Feature Engineering

In [28]:
# DATA CLEANING


# Work on a fresh copy so the original df stays intact for reference
df_clean = df.copy()

print("CLEANING STEPS")
print("-"*40)
print(f"Starting shape: {df_clean.shape}")


# 1. Drop exact duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
removed = before - len(df_clean)
print(f"\n1. Removed {removed} duplicate rows → shape: {df_clean.shape}")


# 2. Cap campaign at 10
capped = (df_clean['campaign'] > 10).sum()
df_clean['campaign'] = df_clean['campaign'].clip(upper=10)
print(f"\n2. Capped 'campaign' at 10. Rows affected: {capped}")


# 3. Transform pdays — create flag and clean numeric
df_clean['was_previously_contacted'] = (df_clean['pdays'] != 999).astype(int)
df_clean['pdays'] = df_clean['pdays'].replace(999, np.nan)
print(f"\n3. Transformed 'pdays':")
print(f"   Previously contacted: {df_clean['was_previously_contacted'].sum():,}")
print(f"   Never contacted:      {(df_clean['was_previously_contacted']==0).sum():,}")


# 4. Make y_binary permanent (replace y with binary version)
df_clean['y_binary'] = (df_clean['y'] == 'yes').astype(int)
print(f"\n4. Created permanent 'y_binary' column")
print(f"   Subscribed:     {df_clean['y_binary'].sum():,}")
print(f"   Not subscribed: {(df_clean['y_binary']==0).sum():,}")


# 5. Standardize categorical strings (safety step)
cat_cols = df_clean.select_dtypes(include='object').columns
for col in cat_cols:
    df_clean[col] = df_clean[col].str.lower().str.strip()
print(f"\n5. Standardized {len(cat_cols)} categorical columns")

CLEANING STEPS
----------------------------------------
Starting shape: (41188, 25)

1. Removed 12 duplicate rows → shape: (41176, 25)

2. Capped 'campaign' at 10. Rows affected: 869

3. Transformed 'pdays':
   Previously contacted: 1,515
   Never contacted:      39,661

4. Created permanent 'y_binary' column
   Subscribed:     4,639
   Not subscribed: 36,537

5. Standardized 11 categorical columns


In [29]:
# Feature engineering — derived features

# Age groups
df_clean['age_group'] = pd.cut(
    df_clean['age'],
    bins=[0, 30, 45, 60, 100],
    labels=['young', 'middle', 'senior', 'retired_age']
)

# Contact intensity buckets
df_clean['contact_intensity'] = pd.cut(
    df_clean['campaign'],
    bins=[0, 1, 3, 5, 10],
    labels=['1_contact', '2-3_contacts', '4-5_contacts', '6-10_contacts']
)

# Disclosure score — how many "unknown" values per row
unknown_check_cols = ['default', 'education', 'housing', 'loan', 'job', 'marital']
df_clean['unknown_count'] = (df_clean[unknown_check_cols] == 'unknown').sum(axis=1)

print(f"Created derived features: age_group, contact_intensity, unknown_count")

Created derived features: age_group, contact_intensity, unknown_count


In [30]:
# Define feature sets for analysis

ALL_FEATURES = [c for c in df_clean.columns if c not in ['y', 'y_binary']]
MODEL_SAFE_FEATURES = [c for c in ALL_FEATURES if c != 'duration']

print(f" Feature sets defined:")
print(f"   ALL_FEATURES:        {len(ALL_FEATURES)} columns")
print(f"   MODEL_SAFE_FEATURES: {len(MODEL_SAFE_FEATURES)} columns (duration excluded)")

 Feature sets defined:
   ALL_FEATURES:        26 columns
   MODEL_SAFE_FEATURES: 25 columns (duration excluded)


In [31]:
# Final summary

print("CLEANING SUMMARY")
print("-"*40)
print(f"Original shape:  {df.shape}")
print(f"Cleaned shape:   {df_clean.shape}")
print(f"New columns added: {set(df_clean.columns) - set(df.columns)}")
print(f"\nFinal columns ({len(df_clean.columns)}):")
print(list(df_clean.columns))

CLEANING SUMMARY
----------------------------------------
Original shape:  (41188, 25)
Cleaned shape:   (41176, 28)
New columns added: {'contact_intensity', 'age_group', 'was_previously_contacted'}

Final columns (28):
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y', 'unknown_count', 'y_binary', 'duration_bucket', 'campaign_bucket', 'was_previously_contacted', 'age_group', 'contact_intensity']
